# Azurity Pharmaceuticals - Prescription Volume Forecasting

This notebook demonstrates an end-to-end ML workflow for pharmaceutical demand forecasting:

1. **Feature Store** - Point-in-time correct features for prescription forecasting
2. **Model Training** - XGBoost regressor for 30-day volume prediction
3. **Model Registry** - Register and version the trained model
4. **Streamlit Visualization** - Interactive A/B test analysis

**Business Value**: Predict prescription volume by product/HCP to optimize inventory, sales rep targeting, and promotional campaigns.

## 1. Setup and Imports

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import *
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

session = get_active_session()
print(f"Connected to Snowflake: {session.get_current_account()}")
print(f"Database: {session.get_current_database()}")
print(f"Schema: {session.get_current_schema()}")

In [ ]:
session.sql("USE DATABASE AZURITY_DEMO_DB").collect()
session.sql("USE SCHEMA ML").collect()
session.sql("USE WAREHOUSE AZURITY_ML_WH").collect()
print("Context set to AZURITY_DEMO_DB.ML")

## 2. Explore Azurity Data

In [ ]:
print("=" * 60)
print("AZURITY PRODUCT PORTFOLIO")
print("=" * 60)
products_df = session.table("RAW.PRODUCTS").to_pandas()
display(products_df)

In [ ]:
print("=" * 60)
print("PRESCRIPTION DATA SUMMARY")
print("=" * 60)

rx_summary = session.sql("""
    SELECT 
        p.PRODUCT_NAME,
        COUNT(DISTINCT r.HCP_ID) AS UNIQUE_HCPS,
        COUNT(*) AS TOTAL_RX,
        SUM(r.QUANTITY) AS TOTAL_UNITS,
        MIN(r.FILL_DATE) AS FIRST_RX,
        MAX(r.FILL_DATE) AS LAST_RX
    FROM RAW.PRESCRIPTIONS r
    JOIN RAW.PRODUCTS p ON r.PRODUCT_ID = p.PRODUCT_ID
    GROUP BY p.PRODUCT_NAME
    ORDER BY TOTAL_UNITS DESC
""").to_pandas()
display(rx_summary)

In [ ]:
print("=" * 60)
print("HEALTHCARE PROVIDER DISTRIBUTION BY DECILE")
print("=" * 60)

hcp_decile = session.sql("""
    SELECT 
        DECILE,
        COUNT(*) AS HCP_COUNT,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS PCT
    FROM RAW.HEALTHCARE_PROVIDERS
    GROUP BY DECILE
    ORDER BY DECILE DESC
""").to_pandas()
print(hcp_decile.to_string(index=False))

## 3. Feature Store Setup

The Snowflake Feature Store provides:
- **Centralized feature management** - Single source of truth for ML features
- **Point-in-time correctness** - ASOF JOIN prevents data leakage in backtesting
- **Automatic refresh** - Features stay up-to-date as source data changes

In [ ]:
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView

fs = FeatureStore(
    session=session,
    database="AZURITY_DEMO_DB",
    name="ML",
    default_warehouse="AZURITY_ML_WH"
)
print("Feature Store initialized")

In [ ]:
product_entity = Entity(name="PRODUCT", join_keys=["PRODUCT_ID"])
hcp_entity = Entity(name="HCP", join_keys=["HCP_ID"])

try:
    fs.register_entity(product_entity)
    print("Registered entity: PRODUCT")
except Exception as e:
    print(f"Entity PRODUCT already exists or error: {e}")

try:
    fs.register_entity(hcp_entity)
    print("Registered entity: HCP")
except Exception as e:
    print(f"Entity HCP already exists or error: {e}")

print("\nRegistered Entities:")
for entity in fs.list_entities().collect():
    print(f"  - {entity['NAME']}")

In [ ]:
rx_features_query = """
SELECT 
    PRODUCT_ID,
    HCP_ID,
    FILL_DATE AS AS_OF_DATE,
    REGION,
    SUM(QUANTITY) OVER (
        PARTITION BY PRODUCT_ID, HCP_ID 
        ORDER BY FILL_DATE 
        ROWS BETWEEN 13 PRECEDING AND CURRENT ROW
    ) AS RX_VOLUME_2W,
    SUM(QUANTITY) OVER (
        PARTITION BY PRODUCT_ID, HCP_ID 
        ORDER BY FILL_DATE 
        ROWS BETWEEN 27 PRECEDING AND CURRENT ROW
    ) AS RX_VOLUME_4W,
    AVG(QUANTITY) OVER (
        PARTITION BY PRODUCT_ID, HCP_ID 
        ORDER BY FILL_DATE 
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS RX_AVG_LAST_WEEK,
    COUNT(*) OVER (
        PARTITION BY PRODUCT_ID, HCP_ID 
        ORDER BY FILL_DATE 
        ROWS BETWEEN 27 PRECEDING AND CURRENT ROW
    ) AS RX_COUNT_4W
FROM RAW.PRESCRIPTIONS
"""

rx_features_df = session.sql(rx_features_query)
print("Created prescription features DataFrame")
rx_features_df.limit(5).show()

In [ ]:
try:
    rx_feature_view = fs.get_feature_view("RX_FEATURES", "v1")
    print("Feature view RX_FEATURES already exists, using existing version")
except:
    rx_feature_view = FeatureView(
        name="RX_FEATURES",
        entities=[product_entity, hcp_entity],
        feature_df=rx_features_df,
        timestamp_col="AS_OF_DATE",
        refresh_freq="1 day",
        desc="Prescription volume features for demand forecasting"
    )
    rx_feature_view = fs.register_feature_view(
        feature_view=rx_feature_view,
        version="v1",
        block=True
    )
    print("Registered Feature View: RX_FEATURES v1")

print("\nFeature View Details:")
print(f"  Name: {rx_feature_view.name}")
print(f"  Version: {rx_feature_view.version}")

## 4. Generate Training Dataset with Point-in-Time Correctness

We use the Feature Store to generate training data that is **point-in-time correct**:
- For each historical date, we only use features that were available at that time
- This prevents data leakage and ensures backtesting accuracy

In [ ]:
spine_query = """
SELECT DISTINCT
    r.PRODUCT_ID,
    r.HCP_ID,
    r.FILL_DATE AS AS_OF_DATE,
    COALESCE(SUM(r2.QUANTITY), 0) AS TARGET_NEXT_30_DAY_VOLUME
FROM RAW.PRESCRIPTIONS r
LEFT JOIN RAW.PRESCRIPTIONS r2 
    ON r.PRODUCT_ID = r2.PRODUCT_ID 
    AND r.HCP_ID = r2.HCP_ID
    AND r2.FILL_DATE > r.FILL_DATE 
    AND r2.FILL_DATE <= DATEADD('day', 30, r.FILL_DATE)
WHERE r.FILL_DATE BETWEEN '2024-06-01' AND '2025-12-01'
GROUP BY r.PRODUCT_ID, r.HCP_ID, r.FILL_DATE
"""

spine_df = session.sql(spine_query)
print(f"Spine DataFrame created")
spine_df.limit(5).show()

In [ ]:
training_data = fs.generate_training_data(
    spine_df=spine_df,
    features=[rx_feature_view],
    spine_timestamp_col="AS_OF_DATE",
    spine_label_cols=["TARGET_NEXT_30_DAY_VOLUME"],
    include_feature_view_timestamp_col=False
)

print("Training data generated with point-in-time correctness")
print(f"Columns: {training_data.columns}")
training_data.limit(5).show()

In [ ]:
training_pdf = training_data.to_pandas()
training_pdf = training_pdf.dropna()

print(f"Training samples: {len(training_pdf):,}")
print(f"Target mean: {training_pdf['TARGET_NEXT_30_DAY_VOLUME'].mean():.2f}")
print(f"Target std: {training_pdf['TARGET_NEXT_30_DAY_VOLUME'].std():.2f}")

## 5. Train XGBoost Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

le_product = LabelEncoder()
le_region = LabelEncoder()

training_pdf['PRODUCT_ENCODED'] = le_product.fit_transform(training_pdf['PRODUCT_ID'])
training_pdf['REGION_ENCODED'] = le_region.fit_transform(training_pdf['REGION'].fillna('Unknown'))

feature_cols = [
    'PRODUCT_ENCODED',
    'REGION_ENCODED',
    'RX_VOLUME_2W',
    'RX_VOLUME_4W',
    'RX_AVG_LAST_WEEK',
    'RX_COUNT_4W'
]

X = training_pdf[feature_cols].fillna(0)
y = training_pdf['TARGET_NEXT_30_DAY_VOLUME']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {len(X_train):,} samples")
print(f"Test set: {len(X_test):,} samples")

In [ ]:
model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print("XGBoost model trained successfully!")

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("=" * 60)
print("MODEL PERFORMANCE METRICS")
print("=" * 60)
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R2 Score: {r2:.4f}")

In [ ]:
print("=" * 60)
print("FEATURE IMPORTANCE")
print("=" * 60)

importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

importance_df['Importance'] = (importance_df['Importance'] * 100).round(1).astype(str) + '%'
print(importance_df.to_string(index=False))

## 6. Register Model in Snowflake Model Registry

The Model Registry provides:
- **Version control** - Track model iterations
- **Metadata management** - Store metrics, comments, tags
- **SQL inference** - Call model.predict() directly from SQL
- **RBAC** - Control access to models with Snowflake roles

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(session=session, database_name="AZURITY_DEMO_DB", schema_name="ML")

model_name = "AZURITY_RX_FORECASTER"
version_name = "v1"

try:
    existing_model = registry.get_model(model_name)
    existing_model.delete_version(version_name)
    print(f"Deleted existing version {version_name}")
except:
    pass

mv = registry.log_model(
    model,
    model_name=model_name,
    version_name=version_name,
    sample_input_data=X_train,
    conda_dependencies=["xgboost", "scikit-learn"],
    metrics={
        "rmse": float(rmse),
        "mae": float(mae),
        "r2": float(r2),
        "training_samples": len(X_train),
        "test_samples": len(X_test)
    },
    comment="Prescription volume forecasting model for Azurity products. Predicts 30-day volume by product/HCP."
)

print("=" * 60)
print("MODEL REGISTERED SUCCESSFULLY")
print("=" * 60)
print(f"Model: AZURITY_DEMO_DB.ML.{model_name}")
print(f"Version: {version_name}")
print(f"\nMetrics:")
for k, v in mv.show_metrics().items():
    print(f"  {k}: {v}")

In [ ]:
print("=" * 60)
print("MODELS IN REGISTRY")
print("=" * 60)
models_df = registry.show_models()
display(models_df[['name', 'comment', 'default_version_name']])

## 7. SQL Inference Demo

Once registered, the model can be called directly from SQL - no Python required!

In [ ]:
inference_result = mv.run(X_test.head(10), function_name="predict")

print("=" * 60)
print("MODEL INFERENCE RESULTS")
print("=" * 60)

comparison = X_test.head(10).copy()
comparison['ACTUAL'] = y_test.head(10).values
comparison['PREDICTED'] = inference_result['output_feature_0'].values
comparison['ERROR'] = abs(comparison['ACTUAL'] - comparison['PREDICTED'])

display(comparison[['RX_VOLUME_2W', 'RX_VOLUME_4W', 'ACTUAL', 'PREDICTED', 'ERROR']])

In [ ]:
print("=" * 60)
print("SQL INFERENCE EXAMPLE")
print("=" * 60)
print("""
-- Run this SQL to score new data:
SELECT 
    PRODUCT_ID,
    HCP_ID,
    AZURITY_RX_FORECASTER!PREDICT(
        PRODUCT_ENCODED,
        REGION_ENCODED,
        RX_VOLUME_2W,
        RX_VOLUME_4W,
        RX_AVG_LAST_WEEK,
        RX_COUNT_4W
    ) AS PREDICTED_30_DAY_VOLUME
FROM AZURITY_DEMO_DB.ML.SCORING_INPUT_VIEW;
""")

## 8. Interactive A/B Test Analysis with Streamlit

Unlike static PowerBI dashboards that use pre-aggregated data, Streamlit cells allow:
- **Dynamic date ranges** - User input drives real-time queries
- **Interactive exploration** - Filter, drill-down, compare on the fly
- **Live data** - Queries Snowflake directly, always fresh

In [ ]:
import streamlit as st
from datetime import date

st.title("Email Campaign A/B Test Analysis")
st.markdown("**Dynamic Analysis**: Adjust date ranges to analyze different campaign periods in real-time.")

col1, col2 = st.columns(2)
with col1:
    start_date = st.date_input("Campaign Start Date", value=date(2025, 1, 1))
with col2:
    end_date = st.date_input("Campaign End Date", value=date(2026, 2, 28))

campaign_results = session.sql(f"""
    SELECT 
        VARIANT,
        COUNT(*) AS EMAILS_SENT,
        SUM(CASE WHEN OPEN_DATE IS NOT NULL THEN 1 ELSE 0 END) AS OPENS,
        SUM(CASE WHEN CLICK_DATE IS NOT NULL THEN 1 ELSE 0 END) AS CLICKS,
        ROUND(SUM(CASE WHEN OPEN_DATE IS NOT NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS OPEN_RATE,
        ROUND(SUM(CASE WHEN CLICK_DATE IS NOT NULL THEN 1 ELSE 0 END) * 100.0 / 
              NULLIF(SUM(CASE WHEN OPEN_DATE IS NOT NULL THEN 1 ELSE 0 END), 0), 2) AS CTR
    FROM RAW.EMAIL_CAMPAIGNS
    WHERE SENT_DATE BETWEEN '{start_date}' AND '{end_date}'
    GROUP BY VARIANT
    ORDER BY VARIANT
""").to_pandas()

st.subheader("Campaign Performance by Variant")

if len(campaign_results) >= 2:
    variant_a = campaign_results[campaign_results['VARIANT'] == 'A'].iloc[0]
    variant_b = campaign_results[campaign_results['VARIANT'] == 'B'].iloc[0]
    
    col1, col2, col3, col4 = st.columns(4)
    with col1:
        st.metric("Variant A Open Rate", f"{variant_a['OPEN_RATE']:.1f}%")
    with col2:
        st.metric("Variant B Open Rate", f"{variant_b['OPEN_RATE']:.1f}%", 
                  delta=f"{variant_b['OPEN_RATE'] - variant_a['OPEN_RATE']:.1f}%")
    with col3:
        st.metric("Variant A CTR", f"{variant_a['CTR']:.1f}%")
    with col4:
        st.metric("Variant B CTR", f"{variant_b['CTR']:.1f}%",
                  delta=f"{variant_b['CTR'] - variant_a['CTR']:.1f}%")
    
    st.subheader("Engagement Comparison")
    chart_data = campaign_results.set_index('VARIANT')[['OPENS', 'CLICKS']]
    st.bar_chart(chart_data)
else:
    st.warning("Not enough data for the selected date range")

st.dataframe(campaign_results)

In [ ]:
import streamlit as st

st.title("Product Performance Trends")

product_options = session.sql("SELECT DISTINCT PRODUCT_NAME FROM RAW.PRODUCTS ORDER BY 1").to_pandas()['PRODUCT_NAME'].tolist()
selected_product = st.selectbox("Select Product", product_options)

product_trend = session.sql(f"""
    SELECT 
        DATE_TRUNC('week', r.FILL_DATE)::DATE AS WEEK,
        SUM(r.QUANTITY) AS WEEKLY_VOLUME,
        COUNT(DISTINCT r.HCP_ID) AS UNIQUE_PRESCRIBERS
    FROM RAW.PRESCRIPTIONS r
    JOIN RAW.PRODUCTS p ON r.PRODUCT_ID = p.PRODUCT_ID
    WHERE p.PRODUCT_NAME = '{selected_product}'
    GROUP BY 1
    ORDER BY 1
""").to_pandas()

st.subheader(f"Weekly Volume: {selected_product}")
st.line_chart(product_trend.set_index('WEEK')['WEEKLY_VOLUME'])

st.subheader(f"Unique Prescribers: {selected_product}")
st.area_chart(product_trend.set_index('WEEK')['UNIQUE_PRESCRIBERS'])

## 9. Summary

This notebook demonstrated:

| Capability | What We Showed |
|------------|----------------|
| **Feature Store** | Created entities, registered feature views with auto-refresh |
| **Point-in-Time** | Generated training data with ASOF JOIN to prevent leakage |
| **Model Training** | XGBoost regressor for 30-day prescription volume |
| **Model Registry** | Registered model with metrics, versioning, SQL inference |
| **Streamlit** | Interactive A/B test analysis with dynamic date ranges |

### Next Steps
1. **MLOps**: Add data drift monitoring with Snowflake ML Observability
2. **Batch Scoring**: Schedule daily inference using Snowflake Tasks
3. **Model Fallback**: Implement automatic rollback on performance degradation

In [ ]:
print("=" * 60)
print("AZURITY ML DEMO COMPLETE")
print("=" * 60)
print(f"\nArtifacts Created:")
print(f"  - Feature Store: AZURITY_DEMO_DB.ML (schema)")
print(f"  - Feature View: RX_FEATURES v1")
print(f"  - Model: AZURITY_DEMO_DB.ML.AZURITY_RX_FORECASTER")
print(f"\nModel Performance:")
print(f"  - RMSE: {rmse:.2f}")
print(f"  - MAE: {mae:.2f}")
print(f"  - R2 Score: {r2:.4f}")